In [6]:
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shreyash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shreyash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
# Reading Dataset
df = pd.read_csv("IMDB Dataset.csv")
print("Dataset Preview:\n", df.head())
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

Dataset Preview:
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [8]:
#preprocessing over database
def preprocess(text):
    text = text.lower()  # lowercase
    
    text = re.sub(r"http\S+", "", text)  
    
    text = text.translate(str.maketrans('', '', string.punctuation)) 
    
    words = text.split()
    
    words = [w for w in words if w not in stop_words]  
    
    words = [lemmatizer.lemmatize(w) for w in words] 
    return " ".join(words)

df["clean_text"] = df["review"].apply(preprocess)
print("\nCleaned Text Sample:\n", df[["review", "clean_text"]].head())

X = df["clean_text"]
y = df["sentiment"]


Cleaned Text Sample:
                                               review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one reviewer mentioned watching 1 oz episode y...  
1  wonderful little production br br filming tech...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#Feature Engineering!
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [4]:
#Model Training 

def evaluate(model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    print("\nModel:", model.__class__.__name__)
    print("Accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))


In [13]:
# Results of Bag OF Words
print("\n--- Using Bag of Words ---")
evaluate(LogisticRegression(max_iter=200), X_train_bow, X_test_bow)
evaluate(MultinomialNB(), X_train_bow, X_test_bow)

print("\n--- Using TF-IDF ---")
evaluate(LogisticRegression(max_iter=200), X_train_tfidf, X_test_tfidf)
evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf)


--- Using Bag of Words ---

Model: LogisticRegression
Accuracy: 0.8845
              precision    recall  f1-score   support

    negative       0.88      0.88      0.88      4949
    positive       0.89      0.89      0.89      5051

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Model: MultinomialNB
Accuracy: 0.8555
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86      4949
    positive       0.87      0.84      0.85      5051

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


--- Using TF-IDF ---

Model: LogisticRegression
Accuracy: 0.8897
              precision    recall  f1-score   support

    negative       0.89      0.88      0.89      4949
    positive       0.89      0.90      0.89      5051

    a